# Main Model — Attention-Gated CycleGAN (Mix of Both Papers)

## Model Architecture

1. **Generators:** Two attention-aware ResNet generators for horse-to-zebra and zebra-to-horse translation.

2. **Attention masks:** Each generator learns two masks using a two-channel softmax:
   - **Foreground mask:** Determines how much of the generated image to use.
   - **Background mask:** Determines how much of the original image to preserve.
   - The two masks sum to 1 at every pixel.

3. **Output blending:** The generated RGB image is combined with the original image according to the attention masks:

   $$
   G(x)=A(x)\odot C(x)+(1-A(x))\odot x
   $$

4. **Discriminators:** Two 70×70 PatchGAN discriminators, one for each image domain.

## Loss Functions

5. **Adversarial loss:** Least-squares GAN (LSGAN) loss to encourage realistic images.

6. **Cycle-consistency loss:** L1 loss to preserve the original image through reconstruction.

7. **Identity loss:** L1 loss to discourage unnecessary image modifications.

## Training Strategy

8. **Optimiser:** Adam for both generators and discriminators.

9. **Replay buffers:** Store up to 50 previously generated images to stabilise discriminator training.

10. **Normalisation:** Instance normalisation with no dropout.

11. **Learning-rate schedule:** CycleGAN-style schedule with an initial constant learning rate followed by gradual decay.

---

## Notes
### Attention Mask Weight Updates

The attention mask is part of the generator, so its parameters are updated to **minimise the generator's total loss**:

$$
\boxed{
\theta_A \leftarrow \theta_A -
\alpha \frac{\partial L_G^{\text{total}}}{\partial \theta_A}
}
$$

Where:
- $\theta_A$: Parameters responsible for generating the attention masks.
- $\alpha$: Learning rate.
- $L_G^{\text{total}}$: Combined generator loss, including adversarial, cycle-consistency and identity losses.

The gradient determines whether each parameter increases or decreases. The actual implementation uses Adam, which adapts the gradient-based updates.

### Identity Loss

Identity loss encourages a generator to leave an image unchanged if it already belongs to the target domain.

$$
\boxed{
\mathcal{L}_{\text{id}} =
\mathbb{E}_{y\sim p_Y}[\|G(y)-y\|_1]
+
\mathbb{E}_{x\sim p_X}[\|F(x)-x\|_1]
}
$$

Where:
- $G$: Horse-to-zebra generator.
- $F$: Zebra-to-horse generator.
- $x$: Real horse image.
- $y$: Real zebra image.

For example, giving an existing zebra image to the horse-to-zebra generator should produce approximately the same zebra image. This discourages unnecessary changes to colour, brightness and appearance.

## 1. Imports and settings

In [ ]:
from pathlib import Path
import random
import itertools
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROOT = Path.cwd()
if not (ROOT / "data" / "horse2zebra").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "horse2zebra"
SAVE = ROOT / "notebooks" / "checkpoints"
SAVE.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 1
EPOCHS = 200            # first 100 constant LR, next 100 linear decay
LR = 0.0002
LAMBDA_CYCLE = 10.0
LAMBDA_IDENTITY = 0.5  # actual identity coefficient = 0.5 * cycle coefficient
IMAGE_SIZE = 256

print("Device:", DEVICE)
print("Dataset:", DATA)


## 2. Unpaired horse/zebra data

One training batch contains **independent** samples from A (horses) and B (zebras). There are no paired ground-truth zebra versions of the horse images. The image transformations help the model generalise to different image positions and orientations.

getitem selects a horse image using its index, wrapping around when necessary. Then randomly chooses a zebra image, independently of the horse. Then applies the transformations and returns both images as tensors.

In [ ]:
class HorseZebraDataset(Dataset):
    def __init__(self, folder, split="train"):
        self.horses = sorted((folder / f"{split}A").glob("*.*"))
        self.zebras = sorted((folder / f"{split}B").glob("*.*"))
        self.transform = transforms.Compose([
            transforms.Resize(286, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomCrop(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.5,) * 3, (0.5,) * 3),
        ])
        assert self.horses and self.zebras, f"No images found in {folder}"

    def __len__(self):
        return max(len(self.horses), len(self.zebras))

    def __getitem__(self, idx):
        a = Image.open(self.horses[idx % len(self.horses)]).convert("RGB")
        b = Image.open(random.choice(self.zebras)).convert("RGB")
        return self.transform(a), self.transform(b)

loader = DataLoader(HorseZebraDataset(DATA), batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=0)
print("Batches per epoch:", len(loader))


## 3. Attention-aware generator with final RGB blending

The network uses the usual ResNet encoder, residual blocks and upsampling decoder. The generator needs to change a horse's appearance without unnecessarily changing its shape, position or surrounding environment.

Residual blocks make it easier to learn transformations while retaining information from earlier layers. They also help gradients flow through deeper networks.

This model uses 9 residual blocks, following the standard CycleGAN ResNet-9 architecture.



Input image (e.g. horse) -> Encoder (reduce spatial dimensions) -> 9 residual blocks -> Decoder (back to 256 x 256) -> Image + Attention mask -> Blend



At the end it has **two heads**:

- `content_head` predicts a complete RGB image (values from -1 to 1).
- `attention_head` predicts **foreground/background masks at full image resolution** (values from 0 to 1, adding to 1).

We blend **the final RGB pixels**, not internal features:

$$
G(x) = A_{fg}(x) \odot C(x) + A_{bg}(x) \odot x
$$

This same structure is used for both horse → zebra and zebra → horse.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)


class AttentionGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.ReflectionPad2d(3), nn.Conv2d(3, 64, 7),
            nn.InstanceNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.InstanceNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.InstanceNorm2d(256), nn.ReLU(inplace=True),
            *[ResidualBlock(256) for _ in range(9)],
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 3, 2, 1, output_padding=1),
            nn.InstanceNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, output_padding=1),
            nn.InstanceNorm2d(64), nn.ReLU(inplace=True),
        )
        # Two predictions from the same decoded features.
        self.content_head = nn.Sequential(
            nn.ReflectionPad2d(3), nn.Conv2d(64, 3, 7), nn.Tanh()
        )
        self.attention_head = nn.Sequential(
            nn.Conv2d(64, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def predict_parts(self, x):
        features = self.decoder(self.encoder(x))
        content = self.content_head(features)    # (batch, 3, H, W)
        masks = self.attention_head(features)    # (batch, 1, H, W)

        foreground = self.attention_head(features)
        background = 1 - foreground
        output = foreground * content + background * x
        
        return output, masks, content

    # basically a shortcut for predict_parts, but only returns the output and masks
    def forward(self, x):
        output, masks, _ = self.predict_parts(x)
        return output, masks


## 4. Discriminator

We use **two 70×70 PatchGAN discriminators**, one for horses and one for zebras.

Unlike a traditional discriminator that outputs one real/fake score, PatchGAN produces a **30×30 grid (900 scores)**. Each score evaluates an overlapping 70×70 region of the original 256×256 image.

### Why 70×70?

The receptive field grows as convolutional layers are stacked:

$$
R_{\text{new}}=R_{\text{old}}+(K-1)J_{\text{old}}
$$

$$
J_{\text{new}}=J_{\text{old}}\times S
$$

Where:
- $R$: Receptive field size.
- $K$: Kernel size.
- $J$: Effective stride relative to the input.
- $S$: Current layer's stride.

Starting with $R=1$ and $J=1$:

| Layer | Kernel | Stride | Receptive field |
|---|---|---|---|
| Conv 1 | 4×4 | 2 | 4×4 |
| Conv 2 | 4×4 | 2 | 10×10 |
| Conv 3 | 4×4 | 2 | 22×22 |
| Conv 4 | 4×4 | 1 | 46×46 |
| Conv 5 | 4×4 | 1 | **70×70** |

For example, the final convolution gives:

$$
R_5=46+(4-1)\times8=\boxed{70}
$$

### Why stack convolutions instead of using one 70×70 filter?

Although a single 70×70 convolution could have the same receptive field, **stacked convolutions with nonlinear activations learn more complex features**, progressing from edges to textures and patterns such as zebra stripes.

### Other Details

- **Output:** 30×30 = 900 real/fake scores, averaged when calculating the adversarial loss.
- **Weight initialisation:** Convolutional weights sampled from $\mathcal{N}(0,0.02^2)$; biases initialised to zero.
- **InstanceNorm2d:** Normalises feature maps to help stabilise training.
- **LeakyReLU:** Preserves gradients for negative activations.

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.InstanceNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.InstanceNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 1, 1), nn.InstanceNorm2d(512), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 1),
        )

    def forward(self, x):
        return self.layers(x)


def init_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

G_AB = AttentionGenerator().to(DEVICE)  # horse -> zebra
G_BA = AttentionGenerator().to(DEVICE)  # zebra -> horse
D_B = PatchDiscriminator().to(DEVICE)   # zebra discriminator
D_A = PatchDiscriminator().to(DEVICE)   # horse discriminator
for network in (G_AB, G_BA, D_B, D_A):
    network.apply(init_weights)
print("Networks initialised")


## 5. Losses and optimiser

- **Least-squares GAN (LSGAN):** generated images should fool the discriminator.
- **Cycle consistency (L1):** `horse → zebra → horse` and the reverse must reconstruct the originals.
- **Identity (L1):** a zebra sent to the horse→zebra generator should stay similar to itself (and vice versa).

Normally, during GAN training, the discriminator receives fake images produced by the generator's current weights. However, the generator is constantly changing. If the discriminator only sees the newest fake images, it may adapt too strongly to the generator's latest outputs. CycleGAN uses an image replay buffer to reduce this instability.

Currently, for the first 50 iterations (since batch size is 1), then after we generate an image, the image is stored in self.images, then the same image gets added to chosen and chosen is returned. Then after 50 iterations, self.images contains 50 images, and when we are pulling a random image (with 50% chance) from self.images, it may be the same as an image already processed.


### Learning-Rate Schedule

We use the CycleGAN learning-rate schedule: **100 epochs at a constant learning rate**, followed by **100 epochs of linear decay**.

The learning-rate multiplier is defined as:

$$
f(e)=
\begin{cases}
1, & e < 100 \\[6pt]
\frac{200-e}{100}, & 100 \leq e \leq 200
\end{cases}
$$

The actual learning rate is:

$$
\boxed{\alpha(e)=0.0002 \times f(e)}
$$

Where:
- $e$: Epoch index.
- $\alpha(e)$: Learning rate at epoch index $e$.
- $0.0002$: Initial learning rate.

This allows larger parameter updates during early training and gradually smaller updates as the model converges.

In [ ]:
mse = nn.MSELoss()                                                              # LSGAN adversarial loss
l1 = nn.L1Loss()                                                                # average absolute difference loss for cycle and identity losses
opt_G = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()), # itertools combines the parameters of both generators into a single sequence.
                         lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(itertools.chain(D_A.parameters(), D_B.parameters()),   # beta_1 controls how much previous gradients influence the current estimate.
                         lr=LR, betas=(0.5, 0.999))                             # beta_2 controls how much previous squared gradients influence the current estimate.

# Store up to 50 previously generated images, as in CycleGAN.
class ImagePool:
    def __init__(self, capacity=50):
        self.capacity = capacity
        self.images = []                                                        # self.images stores up to 50 fake images for future use, and 

    def query(self, batch):                                                     # batch contains the newly generated fake images.
        chosen = []                                                             # chosen will store the images returned to the discriminator.
        for image in batch.detach():                                            # removes the image from PyTorch's computational graph, so discriminator training doesn't backpropagate into the generator.
            image = image.unsqueeze(0)                                          # adds a batch dimension to the image tensor, making it compatible for concatenation later.
            if len(self.images) < self.capacity:
                self.images.append(image.clone())
                chosen.append(image)
            elif random.random() > 0.5:
                index = random.randrange(len(self.images))                      # 50% chance of replacing a previously stored image with the new one and returning the old one.
                chosen.append(self.images[index].clone())
                self.images[index] = image.clone()
            else:                                                               # 50% chance of returning the new one directly.
                chosen.append(image)
        return torch.cat(chosen, dim=0)

pool_A, pool_B = ImagePool(), ImagePool()

# 100 epochs at the original LR, then 100 epochs linearly decay to zero.
def lr_factor(epoch):
    return 1.0 if epoch < 100 else max(0.0, (EPOCHS - epoch) / (EPOCHS - 100))

scheduler_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_factor)
scheduler_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_factor)


## 6. One training iteration

**First update both generators** using the total GAN + cycle + identity loss. **Then update both discriminators**, using real images and detached fake images from the replay pools. Each batch triggers one generator optimiser step and one discriminator optimiser step.

In [ ]:
def train_step(real_A, real_B):
    real_A, real_B = real_A.to(DEVICE), real_B.to(DEVICE)

    # --- Update both generators ---
    opt_G.zero_grad()
    fake_B, _ = G_AB(real_A)                  # horse -> zebra (blended RGB)
    fake_A, _ = G_BA(real_B)                  # zebra -> horse (blended RGB)
    recovered_A, _ = G_BA(fake_B)            # horse -> zebra -> horse
    recovered_B, _ = G_AB(fake_A)            # zebra -> horse -> zebra
    same_B, _ = G_AB(real_B)                 # identity: zebra -> zebra
    same_A, _ = G_BA(real_A)                 # identity: horse -> horse

    pred_fake_B = D_B(fake_B)
    pred_fake_A = D_A(fake_A)
    loss_adv = (mse(pred_fake_B, torch.ones_like(pred_fake_B)) +
                mse(pred_fake_A, torch.ones_like(pred_fake_A)))
    loss_cycle = LAMBDA_CYCLE * (l1(recovered_A, real_A) + l1(recovered_B, real_B))
    loss_identity = LAMBDA_CYCLE * LAMBDA_IDENTITY * (
        l1(same_A, real_A) + l1(same_B, real_B)
    )
    loss_G = loss_adv + loss_cycle + loss_identity
    loss_G.backward()
    opt_G.step()

    # --- Update both discriminators ---
    opt_D.zero_grad()
    past_B = pool_B.query(fake_B)            # detached previous/current fake samples
    past_A = pool_A.query(fake_A)
    pred_real_B, pred_fake_B = D_B(real_B), D_B(past_B)
    pred_real_A, pred_fake_A = D_A(real_A), D_A(past_A)
    loss_D_B = 0.5 * (mse(pred_real_B, torch.ones_like(pred_real_B)) +
                      mse(pred_fake_B, torch.zeros_like(pred_fake_B)))
    loss_D_A = 0.5 * (mse(pred_real_A, torch.ones_like(pred_real_A)) +
                      mse(pred_fake_A, torch.zeros_like(pred_fake_A)))
    loss_D = loss_D_A + loss_D_B
    loss_D.backward()
    opt_D.step()

    return {"G": loss_G.item(), "D": loss_D.item(),
            "GAN": loss_adv.item(), "cycle": loss_cycle.item(),
            "identity": loss_identity.item()}


## 7. Train

This is the long-running cell. First test using `EPOCHS = 1` **before** starting the full 200 epochs (then rerun the optimiser/scheduler cell). The model saves weights every 5 epochs and at the final epoch.

In [ ]:
history = []
for epoch in range(EPOCHS):
    running = {key: 0.0 for key in ("G", "D", "GAN", "cycle", "identity")}
    for horse, zebra in loader:
        losses = train_step(horse, zebra)
        for key in running:
            running[key] += losses[key]
    scheduler_G.step()
    scheduler_D.step()
    avg = {k: v / len(loader) for k, v in running.items()}
    history.append(avg)
    print(f"Epoch {epoch+1:3}/{EPOCHS}: " + ", ".join(f"{k}={v:.3f}" for k, v in avg.items()))

    if (epoch + 1) % 5 == 0 or epoch + 1 == EPOCHS:
        torch.save({"epoch": epoch+1,
                    "G_AB": G_AB.state_dict(), "G_BA": G_BA.state_dict(),
                    "D_A": D_A.state_dict(), "D_B": D_B.state_dict(),
                    "optimizer_G": opt_G.state_dict(), "optimizer_D": opt_D.state_dict(),
                    "scheduler_G": scheduler_G.state_dict(), "scheduler_D": scheduler_D.state_dict()},
                   SAVE / "latest.pt")


## 8. Visualise the input, generated content, mask and final result

This is the easiest way to see what the attention mask is doing. The **candidate RGB image** comes from the content head, and the **foreground mask** selects which pixels are copied from it into the final blended output.

A mask may initially be grey or noisy, before the network has learned useful translations.

In [ ]:
G_AB.eval()
with torch.no_grad():
    horse, _ = next(iter(loader))
    blended_zebra, masks, content = G_AB.predict_parts(horse.to(DEVICE))

def show_rgb(x):
    return ((x[0].detach().cpu().permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(show_rgb(horse)); axes[0].set_title("Original horse")
axes[1].imshow(show_rgb(content)); axes[1].set_title("Candidate zebra RGB")
axes[2].imshow(masks[0, 0].detach().cpu().numpy(), cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Attention mask (white = change)")
axes[3].imshow(show_rgb(blended_zebra)); axes[3].set_title("Final blended zebra")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
G_AB.train()

## Notes for your report

**Model description:** We implemented a *CycleGAN with AttentionGAN-inspired final-image blending*. Each ResNet generator predicts (1) a candidate RGB translation and (2) a foreground/background softmax attention mask at image resolution. The final image is `foreground × candidate + background × input`. Two 70×70 PatchGAN discriminators use least-squares adversarial losses. The model also uses L1 cycle-consistency and identity losses, replay buffers, instance normalisation, Adam and the CycleGAN learning-rate schedule.

**What is original:** A simplified hybrid of the two ideas, with explicit and visualisable per-pixel RGB blending. It is not a reproduction of the authors' full AttentionGAN training procedure.

**Experiment plan:** Compare the existing unchanged CycleGAN baseline with this blended-attention version using the same dataset, image dimensions and comparable training settings. Inspect image quality, backgrounds and masks. Training losses alone do not prove image quality improvement.

**Practical warning:** The masks are learned only indirectly from GAN/cycle/identity losses; there are no ground-truth foreground masks. They might be broad, overly small or semantically imperfect. Inspect results before concluding that they segment horses accurately.

**Run instructions:** Use your `horse2zebra` Conda environment. Start this notebook from the repository root so `data/horse2zebra/trainA` and `trainB` are found. For a first run set `EPOCHS = 1`; return to `200` for full training. Trained checkpoints live in `main_model_checkpoints/`.